# SmolVLA × LIBERO-plus Spatial LoRA Fine-tuning

`lerobot/smolvla_libero_plus`を初期重みとして、
LIBERO-Spatialの10タスクをLoRAで追加学習します。

学習後はLoRAを元モデルへマージし、次の2モデルを
同じLIBERO-plus Spatial環境で比較します。

- 追加学習前のLIBERO-plus重み
- Spatial追加学習後のマージ済みモデル

**既定条件**

- Spatial 10タスク × 各20エピソード
- 3,000 training steps
- 100 stepsごとにlossを表示
- 評価は10タスク × 各3エピソード

Google ColabとローカルJupyterの両方に対応しています。

## 1. 実行環境を確認する

ColabではGPUランタイムを選択してください。ローカルではCUDAまたはApple SiliconのMPSを推奨します。CPUでも実行できますが、学習には非常に長い時間がかかります。

ローカルの作業先は既定で`~/.cache/parc2026_smolvla`です。変更する場合は、このセルの実行前に環境変数`PARC2026_SMOLVLA_WORKDIR`を設定してください。

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

try:
    IS_COLAB = (
        importlib.util.find_spec("google.colab")
        is not None
    )
except ModuleNotFoundError:
    IS_COLAB = False

default_work_dir = (
    Path("/content")
    if IS_COLAB
    else Path.home() / ".cache" / "parc2026_smolvla"
)
WORK_DIR = Path(
    os.environ.get(
        "PARC2026_SMOLVLA_WORKDIR",
        str(default_work_dir),
    )
).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ.setdefault(
    "HF_HOME",
    str(WORK_DIR / "hf_cache"),
)
os.environ.setdefault(
    "HF_LEROBOT_HOME",
    str(WORK_DIR / "lerobot_cache"),
)

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

if torch.cuda.is_available():
    TORCH_DEVICE = "cuda"
    MODEL_TORCH_DTYPE = (
        "bfloat16"
        if torch.cuda.is_bf16_supported()
        else "float16"
    )
    os.environ.setdefault(
        "PYTORCH_CUDA_ALLOC_CONF",
        "expandable_segments:True",
    )
    accelerator_name = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    TORCH_DEVICE = "mps"
    MODEL_TORCH_DTYPE = "float16"
    accelerator_name = "Apple Silicon MPS"
else:
    TORCH_DEVICE = "cpu"
    MODEL_TORCH_DTYPE = "float32"
    accelerator_name = "CPU (学習は非常に低速です)"

MUJOCO_GL_BACKEND = os.environ.get(
    "PARC2026_MUJOCO_GL",
    "egl" if sys.platform.startswith("linux") else "glfw",
)

print(f"Environment: {'Google Colab' if IS_COLAB else 'Local Jupyter'}")
print(f"Accelerator: {accelerator_name}")
print(f"Working directory: {WORK_DIR}")

## 2. システムパッケージを準備する

LeRobot、動画デコード、MuJoCoで必要になるパッケージを確認します。Colabでは自動導入し、ローカルでは事前に導入したコマンドを検査します。

In [ ]:
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


if IS_COLAB:
    run_quiet(["apt-get", "update", "-qq"])
    run_quiet(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "ffmpeg",
            "git",
            "unzip",
            "libgl1",
            "libglib2.0-0",
            "libsm6",
            "libxext6",
            "libexpat1",
            "libfontconfig1-dev",
            "libmagickwand-dev",
        ]
    )
else:
    required_commands = ["ffmpeg", "git", "unzip"]
    missing_commands = [
        command
        for command in required_commands
        if shutil.which(command) is None
    ]
    if missing_commands:
        raise RuntimeError(
            "不足しているシステムコマンド: "
            + ", ".join(missing_commands)
            + "。examples/README.mdのローカル手順を参照してください。"
        )

print("System packages ready.")

## 3. LeRobotをインストールする

LeRobot `v0.6.0`を使用します。
LoRA学習に必要な互換性調整もこのセルで適用します。ローカルでは他のPython環境へ影響させないため、専用venvの利用を推奨します。

In [ ]:
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = WORK_DIR / "lerobot"
LEROBOT_SRC = LEROBOT_DIR / "src"

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

run_quiet(
    [
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ]
)

smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if MODEL_TORCH_DTYPE != "bfloat16":
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        f'torch_dtype="{MODEL_TORCH_DTYPE}",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ]
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

## 4. 学習・評価条件を設定する

Spatialの10タスクから各20エピソードを選び、
合計200エピソードで追加学習します。

評価を正式な10エピソード/taskへ近づける場合は、
`EVAL_EPISODES_PER_TASK = 10`へ変更してください。

In [ ]:
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = (
    "7bb70aa5bc92b82c9239142775d3a173103567ff"
)

VLM_REPO = (
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
)

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = (
    "f3f49f426d75030177b18778374005bc12ccd588"
)

SPATIAL_TASK_NAMES = [
    "pick up the black bowl from table center and place it on the plate",
    "pick up the black bowl next to the cookie box and place it on the plate",
    "pick up the black bowl next to the plate and place it on the plate",
    "pick up the black bowl next to the ramekin and place it on the plate",
    "pick up the black bowl on the cookie box and place it on the plate",
    "pick up the black bowl on the ramekin and place it on the plate",
    "pick up the black bowl on the stove and place it on the plate",
    "pick up the black bowl on the wooden cabinet and place it on the plate",
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the black bowl between the plate and the ramekin and place it on the plate",
]

TRAIN_EPISODES_PER_TASK = 20
GRIPPER_QUALITY_TASK_IDS = set(range(len(SPATIAL_TASK_NAMES)))
MIN_GRIPPER_RUN_FRAMES = 3

STEPS = 3000
LOG_FREQ = 100
BATCH_SIZE = 1
LEARNING_RATE = 3e-4
FINAL_LEARNING_RATE = 3e-5
WARMUP_STEPS = 100
LORA_R = 16
LORA_ALPHA = 16
SEED = 42

EVAL_TASK_IDS = list(range(10))
EVAL_EPISODES_PER_TASK = 3
EVAL_SEED = 2026

OUTPUT_DIR = (
    WORK_DIR / "outputs" / "smolvla_libero_plus_spatial_lora"
)
MERGED_MODEL_DIR = (
    WORK_DIR / "smolvla_libero_plus_spatial_lora_merged"
)
BASELINE_MODEL_DIR = (
    WORK_DIR / "smolvla_libero_plus_baseline"
)

BASE_EVAL_DIR = WORK_DIR / "eval" / "base"
FINETUNED_EVAL_DIR = WORK_DIR / "eval" / "spatial_lora"
COMPARISON_CSV_PATH = (
    WORK_DIR / "libero_spatial_comparison.csv"
)
SUBMISSION_ZIP_PATH = (
    WORK_DIR / "PARC2026_track1_submission.zip"
)
SUBMISSION_RUNTIME_REVISION = (
    "f995b13df29275a98e16651cad4a314246ed03b2"
)

MIXED_PRECISION = (
    "bf16" if MODEL_TORCH_DTYPE == "bfloat16"
    else "fp16" if MODEL_TORCH_DTYPE == "float16"
    else "no"
)

## 5. 公開ファイルの取得処理を用意する

キャッシュを優先し、匿名アクセスの制限時は自動的に再試行します。

In [ ]:
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Faceからの取得に失敗しました。"
    ) from last_error


def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 6. Spatial学習データを選ぶ

actionとstateが異なる行動軌跡から各20エピソードを選択します。
グリッパー操作の品質判定は全10タスクに適用します。
選定結果は`training_episode_selection.json`へ保存します。

In [ ]:
# BEGIN_TRAJECTORY_SELECTION_HELPERS
import hashlib
import json
import re
from collections import defaultdict

import numpy as np


def build_episode_trajectories(
    frame_episode_indices: np.ndarray,
    actions: np.ndarray,
    states: np.ndarray,
) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    episode_indices = np.asarray(frame_episode_indices)
    action_values = np.asarray(actions)
    state_values = np.asarray(states)

    if episode_indices.ndim != 1:
        raise ValueError(
            "frame_episode_indices must be one-dimensional."
        )
    if action_values.ndim != 2 or state_values.ndim != 2:
        raise ValueError(
            "actions and states must be two-dimensional."
        )
    if not (
        len(episode_indices)
        == len(action_values)
        == len(state_values)
    ):
        raise ValueError("Frame counts do not match.")
    if len(episode_indices) == 0:
        return {}

    boundaries = np.flatnonzero(
        episode_indices[1:] != episode_indices[:-1]
    ) + 1
    starts = np.concatenate(([0], boundaries))
    ends = np.concatenate((boundaries, [len(episode_indices)]))
    trajectories = {}

    for start, end in zip(starts, ends, strict=True):
        episode_index = int(episode_indices[start])
        if episode_index in trajectories:
            raise ValueError(
                f"Episode {episode_index} frames are not contiguous."
            )
        trajectories[episode_index] = (
            action_values[start:end],
            state_values[start:end],
        )

    return trajectories


def trajectory_fingerprint(
    actions: np.ndarray,
    states: np.ndarray,
) -> str:
    action_values = np.asarray(actions, dtype="<f4", order="C")
    state_values = np.asarray(states, dtype="<f4", order="C")

    if action_values.ndim != 2 or state_values.ndim != 2:
        raise ValueError(
            "actions and states must be two-dimensional."
        )
    if (
        len(action_values) == 0
        or len(action_values) != len(state_values)
    ):
        raise ValueError("The trajectory length is invalid.")
    if (
        not np.isfinite(action_values).all()
        or not np.isfinite(state_values).all()
    ):
        raise ValueError("The trajectory contains NaN or Inf.")

    digest = hashlib.sha256()
    digest.update(
        np.asarray(action_values.shape, dtype="<i8").tobytes()
    )
    digest.update(action_values.tobytes())
    digest.update(
        np.asarray(state_values.shape, dtype="<i8").tobytes()
    )
    digest.update(state_values.tobytes())
    return digest.hexdigest()


def evaluate_gripper_quality(
    actions: np.ndarray,
    *,
    min_run_frames: int = 3,
) -> dict:
    action_values = np.asarray(actions)
    if action_values.ndim != 2 or action_values.shape[1] < 1:
        raise ValueError(
            "actions must include a gripper dimension."
        )
    if min_run_frames < 1:
        raise ValueError("min_run_frames must be positive.")

    gripper = action_values[:, -1]
    if len(gripper) == 0 or not np.isfinite(gripper).all():
        raise ValueError("Gripper actions are empty or invalid.")
    if np.any(gripper == 0):
        return {
            "passed": False,
            "run_values": [],
            "run_lengths": [],
            "reason": "Gripper action contains zero.",
        }

    signs = np.where(gripper > 0, 1, -1)
    boundaries = np.flatnonzero(signs[1:] != signs[:-1]) + 1
    starts = np.concatenate(([0], boundaries))
    ends = np.concatenate((boundaries, [len(signs)]))
    run_values = [int(value) for value in signs[starts]]
    run_lengths = [
        int(end - start)
        for start, end in zip(starts, ends, strict=True)
    ]
    expected = [-1, 1, -1]

    if run_values != expected:
        if run_values == [-1, 1]:
            reason = "Missing release after grasp."
        elif len(run_values) > len(expected):
            reason = "Extra gripper reversals detected."
        else:
            reason = "Expected open-close-open gripper sequence."
        return {
            "passed": False,
            "run_values": run_values,
            "run_lengths": run_lengths,
            "reason": reason,
        }

    if min(run_lengths) < min_run_frames:
        return {
            "passed": False,
            "run_values": run_values,
            "run_lengths": run_lengths,
            "reason": (
                f"A gripper run is shorter than {min_run_frames} frames."
            ),
        }

    return {
        "passed": True,
        "run_values": run_values,
        "run_lengths": run_lengths,
        "reason": None,
    }


def evenly_spaced_positions(length: int, count: int) -> list[int]:
    if count < 1:
        raise ValueError("count must be positive.")
    if length < count:
        raise ValueError(
            f"Only {length} distinct trajectories for {count} selections."
        )
    if count == 1:
        return [length // 2]

    return [
        round(index * (length - 1) / (count - 1))
        for index in range(count)
    ]


def select_episodes_by_trajectory(
    episode_indices: list[int],
    trajectory_by_episode: dict[
        int, tuple[np.ndarray, np.ndarray]
    ],
    count: int,
    *,
    require_gripper_quality: bool = False,
    min_gripper_run_frames: int = 3,
) -> dict:
    accepted = []
    rejections = []

    for episode_index in episode_indices:
        actions, states = trajectory_by_episode[episode_index]
        if require_gripper_quality:
            quality = evaluate_gripper_quality(
                actions,
                min_run_frames=min_gripper_run_frames,
            )
            if not quality["passed"]:
                rejections.append(
                    {
                        "episode_index": episode_index,
                        "reason": quality["reason"],
                    }
                )
                continue
        accepted.append((episode_index, actions, states))

    grouped = defaultdict(list)
    for episode_index, actions, states in accepted:
        grouped[trajectory_fingerprint(actions, states)].append(
            episode_index
        )

    fingerprints = list(grouped)
    selected_fingerprints = [
        fingerprints[position]
        for position in evenly_spaced_positions(
            len(fingerprints), count
        )
    ]
    selected_episode_indices = []

    for rank, fingerprint in enumerate(selected_fingerprints):
        variants = grouped[fingerprint]
        variant_position = (
            len(variants) // 2
            if count == 1
            else round(
                rank * (len(variants) - 1) / (count - 1)
            )
        )
        selected_episode_indices.append(
            variants[variant_position]
        )

    return {
        "selected_episode_indices": selected_episode_indices,
        "selected_trajectory_fingerprints": selected_fingerprints,
        "candidate_count": len(episode_indices),
        "quality_candidate_count": len(accepted),
        "unique_trajectory_count": len(grouped),
        "quality_rejections": rejections,
    }
# END_TRAJECTORY_SELECTION_HELPERS

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)
from lerobot.datasets.lerobot_dataset import LeRobotDataset


def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in zip(
    dataset_metadata.episodes["episode_index"],
    dataset_metadata.episodes["tasks"],
    strict=True,
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}
actual_spatial_tasks = []

for task_name in SPATIAL_TASK_NAMES:
    actual_task = available_by_normalized.get(
        normalize_task_name(task_name)
    )
    if actual_task is None:
        raise RuntimeError(
            f"Spatial task not found: {task_name}"
        )
    actual_spatial_tasks.append(actual_task)

all_spatial_episode_indices = sorted(
    episode_index
    for task_name in actual_spatial_tasks
    for episode_index in task_to_episodes[task_name]
)
trajectory_dataset = run_hf_with_retry(
    lambda: LeRobotDataset(
        DATASET_REPO,
        episodes=all_spatial_episode_indices,
        revision=DATASET_REVISION,
        download_videos=False,
    )
)
trajectory_table = trajectory_dataset.hf_dataset.data
trajectory_by_episode = build_episode_trajectories(
    np.asarray(
        trajectory_table.column("episode_index").to_pylist(),
        dtype=np.int64,
    ),
    np.asarray(
        trajectory_table.column("action").to_pylist(),
        dtype=np.float32,
    ),
    np.asarray(
        trajectory_table.column("observation.state").to_pylist(),
        dtype=np.float32,
    ),
)

selected_by_task = {}
selection_manifest = {
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "strategy": "action_state_trajectory_fingerprint",
    "episodes_per_task": TRAIN_EPISODES_PER_TASK,
    "gripper_quality_task_ids": sorted(
        GRIPPER_QUALITY_TASK_IDS
    ),
    "min_gripper_run_frames": MIN_GRIPPER_RUN_FRAMES,
    "tasks": [],
}

for task_id, task_name in enumerate(actual_spatial_tasks):
    selection = select_episodes_by_trajectory(
        task_to_episodes[task_name],
        trajectory_by_episode,
        TRAIN_EPISODES_PER_TASK,
        require_gripper_quality=(
            task_id in GRIPPER_QUALITY_TASK_IDS
        ),
        min_gripper_run_frames=MIN_GRIPPER_RUN_FRAMES,
    )
    selected_by_task[task_name] = selection[
        "selected_episode_indices"
    ]
    selection_manifest["tasks"].append(
        {
            "task_id": task_id,
            "task": task_name,
            "gripper_quality_enabled": (
                task_id in GRIPPER_QUALITY_TASK_IDS
            ),
            **selection,
        }
    )
    print(
        f"Task {task_id}: "
        f"candidates={selection['candidate_count']}, "
        f"quality={selection['quality_candidate_count']}, "
        f"unique={selection['unique_trajectory_count']}, "
        f"selected={selection['selected_episode_indices']}"
    )

EPISODE_INDICES = sorted(
    episode_index
    for episode_indices in selected_by_task.values()
    for episode_index in episode_indices
)
expected_episode_count = (
    len(SPATIAL_TASK_NAMES) * TRAIN_EPISODES_PER_TASK
)
if len(EPISODE_INDICES) != expected_episode_count:
    raise RuntimeError("Episode selection failed.")

selection_manifest_path = (
    WORK_DIR / "training_episode_selection.json"
)
selection_manifest_path.write_text(
    json.dumps(
        selection_manifest,
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

del trajectory_dataset
del trajectory_table
del trajectory_by_episode

print(
    f"Training data: {len(SPATIAL_TASK_NAMES)} tasks × "
    f"{TRAIN_EPISODES_PER_TASK} episodes = "
    f"{len(EPISODE_INDICES)} episodes"
)
print(f"Selection manifest: {selection_manifest_path}")

## 7. 初期重みを準備する

In [ ]:
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "train_config.json",
        "policy_preprocessor.json",
        "policy_preprocessor*.safetensors",
        "policy_postprocessor.json",
        "policy_postprocessor*.safetensors",
    ],
    ignore_patterns=[
        "README.md",
        "eval/**",
    ],
)

if not (
    BASE_MODEL_LOCAL / "model.safetensors"
).is_file():
    raise FileNotFoundError("Base model not found.")

print("Base model ready.")

## 8. LoRA学習を実行する

100 stepごとに平均lossとlearning rateを表示します。

In [ ]:
import re
from collections import deque

episodes_json = (
    "["
    + ",".join(map(str, EPISODE_INDICES))
    + "]"
)

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.device={TORCH_DEVICE}",
    f"--policy.vlm_model_name={VLM_REPO}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=smolvla_libero_plus_spatial_lora",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=0",
    "--persistent_workers=false",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={STEPS}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

training_env = os.environ.copy()
training_env["PYTHONPATH"] = (
    str(LEROBOT_SRC)
    + os.pathsep
    + training_env.get("PYTHONPATH", "")
)
training_env["ACCELERATE_MIXED_PRECISION"] = (
    MIXED_PRECISION
)
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print("Preparing data and starting training...")

process = subprocess.Popen(
    command,
    cwd=LEROBOT_DIR,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

recent_lines: deque[str] = deque(maxlen=80)
report_step = LOG_FREQ

assert process.stdout is not None

for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()

    if not line:
        continue

    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        loss_match = re.search(
            r"loss:([0-9.eE+-]+)",
            line,
        )
        lr_match = re.search(
            r"lr:([0-9.eE+-]+)",
            line,
        )

        loss = (
            loss_match.group(1)
            if loss_match
            else "n/a"
        )
        lr = (
            lr_match.group(1)
            if lr_match
            else "n/a"
        )

        print(
            f"step {report_step:4d}/{STEPS}  "
            f"loss={loss}  lr={lr}"
        )
        report_step += LOG_FREQ

return_code = process.wait()

if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(
        f"Training failed: {return_code}"
    )

print("Training complete.")

## 9. LoRAをマージしてモデル全体を保存する

LoRA差分を元weightへ統合し、通常のLeRobotモデルとして保存します。

In [ ]:
import contextlib
import gc
import io
import json

from peft import PeftModel
from transformers import AutoConfig, AutoTokenizer
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

checkpoint_dir = (
    OUTPUT_DIR
    / "checkpoints"
    / f"{STEPS:06d}"
    / "pretrained_model"
)

if not (
    checkpoint_dir / "adapter_model.safetensors"
).is_file():
    raise FileNotFoundError("Final adapter not found.")

gc.collect()
torch.cuda.empty_cache()

merge_config = PreTrainedConfig.from_pretrained(
    checkpoint_dir
)
merge_config.device = "cpu"
merge_config.pretrained_path = BASE_MODEL_LOCAL
merge_config.use_peft = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    base_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=merge_config,
        strict=False,
    )

    peft_policy = PeftModel.from_pretrained(
        base_policy,
        checkpoint_dir,
        is_trainable=False,
        torch_device="cpu",
    )

    merged_policy = peft_policy.merge_and_unload(
        safe_merge=True
    )

shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO

merged_policy.save_pretrained(MERGED_MODEL_DIR)

vlm_assets_dir = MERGED_MODEL_DIR / "vlm_assets"
vlm_assets_dir.mkdir(parents=True, exist_ok=True)
AutoConfig.from_pretrained(VLM_REPO).save_pretrained(
    vlm_assets_dir
)
AutoTokenizer.from_pretrained(VLM_REPO).save_pretrained(
    vlm_assets_dir
)

for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(
            source_path,
            MERGED_MODEL_DIR / source_path.name,
        )

merged_weights_path = (
    MERGED_MODEL_DIR / "model.safetensors"
)

with safe_open(
    merged_weights_path,
    framework="pt",
    device="cpu",
) as weights:
    if any(
        "lora_" in key.lower()
        for key in weights.keys()
    ):
        raise RuntimeError(
            "LoRA parameters remain after merge."
        )

del peft_policy
del base_policy
del merged_policy

gc.collect()
torch.cuda.empty_cache()

print("Merged model ready.")

## 10. 比較用ベースラインを準備する

公開weightを追加学習モデルと同じ入力schema・processorへ揃えます。

In [ ]:
baseline_config = PreTrainedConfig.from_pretrained(
    MERGED_MODEL_DIR
)
baseline_config.device = "cpu"
baseline_config.pretrained_path = BASE_MODEL_LOCAL
baseline_config.use_peft = False
baseline_config.load_vlm_weights = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    baseline_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=baseline_config,
        strict=False,
    )

shutil.rmtree(BASELINE_MODEL_DIR, ignore_errors=True)
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

baseline_policy.config.use_peft = False
baseline_policy.config.pretrained_path = None
baseline_policy.config.push_to_hub = False
baseline_policy.config.repo_id = None
baseline_policy.config.device = None
baseline_policy.config.load_vlm_weights = False
baseline_policy.config.vlm_model_name = VLM_REPO
baseline_policy.save_pretrained(BASELINE_MODEL_DIR)

for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in MERGED_MODEL_DIR.glob(pattern):
        shutil.copy2(
            source_path,
            BASELINE_MODEL_DIR / source_path.name,
        )

del baseline_policy
gc.collect()
torch.cuda.empty_cache()

print("Baseline ready.")

## 11. LIBERO-plus評価環境を準備する

MuJoCo、LIBERO-plus fork、評価assetsを導入します。

In [ ]:
from huggingface_hub import hf_hub_download

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = WORK_DIR / "LIBERO-plus"
LIBERO_PLUS_PACKAGE_ROOT = (
    LIBERO_PLUS_DIR / "libero" / "libero"
)
LIBERO_PLUS_ASSETS_DIR = (
    LIBERO_PLUS_PACKAGE_ROOT / "assets"
)

os.environ["MUJOCO_GL"] = MUJOCO_GL_BACKEND

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "hf-libero",
        "libero",
        "robosuite",
    ],
    check=False,
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "robosuite==1.4.1",
        "bddl==1.0.1",
        "easydict==1.13",
        "mujoco==3.7.0",
        "matplotlib==3.10.8",
        "Wand==0.6.13",
        "scikit-image==0.25.2",
        "gym==0.26.2",
    ]
)

if (
    importlib.metadata.version("robosuite")
    != "1.4.1"
):
    raise RuntimeError(
        "robosuite 1.4.1 is required."
    )

if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(
        LIBERO_PLUS_DIR,
        ignore_errors=True,
    )
    run_quiet(
        [
            "git",
            "clone",
            "--quiet",
            "https://github.com/sylvestf/LIBERO-plus.git",
            str(LIBERO_PLUS_DIR),
        ]
    )

checkout = run_quiet(
    [
        "git",
        "-C",
        str(LIBERO_PLUS_DIR),
        "checkout",
        "--quiet",
        LIBERO_PLUS_SHA,
    ],
    check=False,
)

if checkout.returncode != 0:
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "fetch",
            "--quiet",
            "--depth",
            "1",
            "origin",
            LIBERO_PLUS_SHA,
        ]
    )
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ]
    )

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "-e",
        str(LIBERO_PLUS_DIR),
    ]
)

if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = WORK_DIR / "libero_plus_assets"
    archive_path = Path(
        run_hf_with_retry(
            lambda: hf_hub_download(
                repo_id="Sylvest/LIBERO-plus",
                repo_type="dataset",
                filename="assets.zip",
                local_dir=assets_root,
                token=False,
            )
        )
    )
    extract_dir = assets_root / "extract"

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    run_quiet(
        [
            "unzip",
            "-q",
            str(archive_path),
            "-d",
            str(extract_dir),
        ]
    )

    candidates = sorted(
        [
            path
            for path in extract_dir.rglob("assets")
            if path.is_dir()
        ],
        key=lambda path: len(path.parts),
    )

    if not candidates:
        raise FileNotFoundError(
            "LIBERO-plus assets not found."
        )

    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    shutil.move(
        str(candidates[0]),
        str(LIBERO_PLUS_ASSETS_DIR),
    )
    shutil.rmtree(assets_root, ignore_errors=True)

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(
    parents=True,
    exist_ok=True,
)
libero_config_path = libero_config_dir / "config.yaml"
libero_config_backup_path = (
    libero_config_dir / "config.yaml.bak"
)
if (
    not IS_COLAB
    and libero_config_path.is_file()
    and not libero_config_backup_path.exists()
):
    shutil.copy2(
        libero_config_path,
        libero_config_backup_path,
    )
    print(
        "既存のLIBERO設定をバックアップしました: "
        f"{libero_config_backup_path}"
    )

libero_config_path.write_text(
    "\n".join(
        [
            f"assets: {LIBERO_PLUS_ASSETS_DIR}",
            (
                "bddl_files: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
            ),
            (
                "datasets: "
                f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
            ),
            (
                "init_states: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)

eval_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_eval.py"
)
source = eval_script.read_text(encoding="utf-8")

source = source.replace(
    "logging.info(pformat(asdict(cfg)))",
    "logging.debug(pformat(asdict(cfg)))",
    1,
)
source = source.replace(
    "max_episodes_rendered = 0 if cfg.eval.recording else 10",
    "max_episodes_rendered = 0",
    1,
)
source = source.replace(
    "disable=inside_slurm()",
    "disable=True",
)

progress_state = (
    '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
)
if progress_state not in source:
    import_anchor = "from tqdm import trange\n"
    if import_anchor not in source:
        raise RuntimeError(
            "Evaluation progress import anchor not found."
        )
    source = source.replace(
        import_anchor,
        import_anchor + "\n" + progress_state + "\n",
        1,
    )

task_loop_anchor = (
    "        for i, (task_group, task_id, env) "
    "in enumerate(tasks):\n"
)
task_loop_patch = (
    task_loop_anchor
    + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
    + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
)
if (
    '_EVAL_PROGRESS["task_index"] = i + 1'
    not in source
):
    if task_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation task-loop anchor not found."
        )
    source = source.replace(
        task_loop_anchor,
        task_loop_patch,
        1,
    )

episode_loop_anchor = "    for batch_ix in progbar:\n"
episode_progress_line = (
    '        print('
    'f"EVAL_PROGRESS '
    "task={_EVAL_PROGRESS['task_index']}/"
    "{_EVAL_PROGRESS['task_total']} "
    'episode={batch_ix + 1}/{n_batches}", '
    "flush=True)\n"
)
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation episode-loop anchor not found."
        )
    source = source.replace(
        episode_loop_anchor,
        episode_loop_anchor + episode_progress_line,
        1,
    )

eval_script.write_text(
    source,
    encoding="utf-8",
)

libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [
    item
    for item in sys.path
    if item != libero_plus_path
]
sys.path.insert(0, libero_plus_path)

for module_name in list(sys.modules):
    if (
        module_name == "libero"
        or module_name.startswith("libero.")
        or module_name == "robosuite"
        or module_name.startswith("robosuite.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

import libero
from libero.libero import benchmark

search_paths = [
    Path(path).resolve()
    for path in getattr(libero, "__path__", [])
]

if not any(
    LIBERO_PLUS_DIR.resolve() in path.parents
    or path == LIBERO_PLUS_DIR.resolve()
    for path in search_paths
):
    raise RuntimeError(
        "LIBERO-plus fork was not loaded."
    )

benchmark_path = Path(
    benchmark.__file__
).resolve()

if (
    LIBERO_PLUS_DIR.resolve()
    not in benchmark_path.parents
):
    raise RuntimeError(
        "LIBERO-plus benchmark was not loaded."
    )

print("LIBERO-plus ready.")

## 12. 追加学習前後を評価する

追加学習前後の2モデルを、同じ10タスク・同じseedで評価します。
評価は1モデルにつき30 rollout、2モデル合計で60 rolloutです。

In [ ]:
import json
import re
from collections import deque

EVAL_CAMERA_MAPPING = {
    "agentview_image": "front",
    "robot0_eye_in_hand_image": "wrist",
}


def build_eval_command(
    policy_path: Path,
    output_dir: Path,
) -> list[str]:
    return [
        "lerobot-eval",
        f"--policy.path={policy_path}",
        f"--policy.device={TORCH_DEVICE}",
        "--policy.use_amp=false",
        "--env.type=libero",
        "--env.is_libero_plus=true",
        "--env.task=libero_spatial",
        (
            "--env.task_ids="
            + json.dumps(
                EVAL_TASK_IDS,
                separators=(",", ":"),
            )
        ),
        (
            "--env.camera_name_mapping="
            + json.dumps(
                EVAL_CAMERA_MAPPING,
                separators=(",", ":"),
            )
        ),
        "--env.observation_height=256",
        "--env.observation_width=256",
        "--env.control_mode=relative",
        "--env.max_parallel_tasks=1",
        "--eval.batch_size=1",
        (
            "--eval.n_episodes="
            f"{EVAL_EPISODES_PER_TASK}"
        ),
        "--eval.use_async_envs=false",
        "--eval.recording=false",
        f"--seed={EVAL_SEED}",
        f"--output_dir={output_dir}",
    ]


def run_evaluation(
    policy_path: Path,
    output_dir: Path,
    label: str,
) -> dict:
    shutil.rmtree(
        output_dir,
        ignore_errors=True,
    )

    eval_env = os.environ.copy()
    eval_env["MUJOCO_GL"] = MUJOCO_GL_BACKEND
    eval_env["PYTHONPATH"] = (
        str(LIBERO_PLUS_DIR)
        + os.pathsep
        + str(LEROBOT_SRC)
        + os.pathsep
        + eval_env.get("PYTHONPATH", "")
    )
    eval_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_HUB_VERBOSITY"] = "error"
    eval_env["TQDM_DISABLE"] = "1"
    eval_env["PYTHONWARNINGS"] = "ignore"
    eval_env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        build_eval_command(
            policy_path,
            output_dir,
        ),
        cwd=LEROBOT_DIR,
        env=eval_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    recent_lines: deque[str] = deque(
        maxlen=120
    )

    progress_pattern = re.compile(
        r"^EVAL_PROGRESS "
        r"task=(\d+)/(\d+) "
        r"episode=(\d+)/(\d+)$"
    )

    assert process.stdout is not None

    for raw_line in process.stdout:
        line = (
            raw_line
            .replace("\r", "")
            .strip()
        )

        if not line:
            continue

        recent_lines.append(line)
        match = progress_pattern.match(line)

        if match:
            (
                task_index,
                task_total,
                episode_index,
                episode_total,
            ) = match.groups()

            print(
                f"{label:<13} | "
                f"task {task_index}/{task_total} | "
                f"episode {episode_index}/{episode_total}"
            )

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "\n".join(recent_lines)
        )

    result_path = (
        output_dir
        / "eval_info.json"
    )

    if not result_path.is_file():
        raise FileNotFoundError(
            result_path
        )

    return json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


BASE_EVAL_INFO = run_evaluation(
    BASELINE_MODEL_DIR,
    BASE_EVAL_DIR,
    "Base model",
)

FINETUNED_EVAL_INFO = run_evaluation(
    MERGED_MODEL_DIR,
    FINETUNED_EVAL_DIR,
    "Spatial LoRA",
)

print("Evaluation complete.")

## 13. 成功率を比較する

`Δ (pp)`は、追加学習後から追加学習前を引いた成功率差です。

In [ ]:
import pandas as pd
from IPython.display import display


def per_task_success(
    eval_info: dict,
) -> dict[int, float]:
    result: dict[int, float] = {}

    for task_info in eval_info["per_task"]:
        task_id = int(task_info["task_id"])
        successes = task_info["metrics"]["successes"]
        result[task_id] = (
            100.0
            * sum(bool(value) for value in successes)
            / len(successes)
        )

    return result


base_per_task = per_task_success(BASE_EVAL_INFO)
finetuned_per_task = per_task_success(
    FINETUNED_EVAL_INFO
)

rows = []

for task_id in EVAL_TASK_IDS:
    base_score = base_per_task[task_id]
    finetuned_score = finetuned_per_task[task_id]

    rows.append(
        {
            "Task ID": task_id,
            "Task": SPATIAL_TASK_NAMES[task_id],
            "Base (%)": base_score,
            "Spatial LoRA (%)": finetuned_score,
            "Δ (pp)": finetuned_score - base_score,
        }
    )

base_overall = float(
    BASE_EVAL_INFO["overall"]["pc_success"]
)
finetuned_overall = float(
    FINETUNED_EVAL_INFO["overall"]["pc_success"]
)

rows.append(
    {
        "Task ID": "Overall",
        "Task": "LIBERO-Spatial",
        "Base (%)": base_overall,
        "Spatial LoRA (%)": finetuned_overall,
        "Δ (pp)": finetuned_overall - base_overall,
    }
)

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(
    COMPARISON_CSV_PATH,
    index=False,
)

display(comparison_df.round(1))

print(
    f"Overall: {base_overall:.1f}% → "
    f"{finetuned_overall:.1f}% "
    f"({finetuned_overall - base_overall:+.1f} pp)"
)

## 14. 提出ZIPと比較結果をダウンロードする

このセルはマージ済みモデルにポリシーサーバーとオフライン実行環境を追加し、
採点へそのままアップロードできる`PARC2026_track1_submission.zip`を作ります。

In [ ]:
import urllib.request

runtime_dir = WORK_DIR / "parc2026_submission_runtime"
shutil.rmtree(runtime_dir, ignore_errors=True)
runtime_dir.mkdir(parents=True)

repository_raw_url = (
    "https://raw.githubusercontent.com/"
    "libkazz/PARC2026_pre/"
    f"{SUBMISSION_RUNTIME_REVISION}"
)
runtime_files = {
    "build_submission.py": (
        "examples/smolvla_policy_server/build_submission.py"
    ),
    "policy_server.py": (
        "examples/smolvla_policy_server/policy_server.py"
    ),
    "smolvla_policy.py": (
        "examples/smolvla_policy_server/smolvla_policy.py"
    ),
    "requirements.txt": (
        "examples/smolvla_policy_server/requirements.txt"
    ),
    "validate_submission.py": "validate_submission.py",
}

for local_name, repository_path in runtime_files.items():
    url = f"{repository_raw_url}/{repository_path}"
    with urllib.request.urlopen(
        url,
        timeout=60,
    ) as response:
        (runtime_dir / local_name).write_bytes(
            response.read()
        )

run_quiet(
    [
        sys.executable,
        str(runtime_dir / "build_submission.py"),
        "--model-dir",
        str(MERGED_MODEL_DIR),
        "--output",
        str(SUBMISSION_ZIP_PATH),
    ]
)
validation = run_quiet(
    [
        sys.executable,
        str(runtime_dir / "validate_submission.py"),
        str(SUBMISSION_ZIP_PATH),
        "--static",
    ]
)
print(validation.stdout)
print(f"Submission ready: {SUBMISSION_ZIP_PATH}")
print(f"Comparison: {COMPARISON_CSV_PATH}")

if IS_COLAB:
    from google.colab import files

    files.download(str(SUBMISSION_ZIP_PATH))
    files.download(str(COMPARISON_CSV_PATH))
else:
    print("ローカル実行のため、上記のパスへ保存しました。")